In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os

os.environ["OPENCV_LOG_LEVEL"] = "ERROR"

In [3]:
from pathlib import Path

import numpy as np
import pandas as pd
from jppype import Mosaic, vscode_theme
from tqdm import tqdm, trange

from fundus_toolkits import FundusData
from fundus_vessels_toolkit.models.topology.dataset import BranchDigraphDataset, SampleInfo
from fundus_vessels_toolkit.segment_to_graph.av_map_fixing import rasterize_tree
from fundus_vessels_toolkit.segment_to_graph.graph_simplification import simplify_passing_nodes
from fundus_vessels_toolkit.segment_to_graph.tree_topology import optimal_topology, transfer_topology
from fundus_vessels_toolkit.utils.jppype import AV_COLORS, draw_graph, draw_tree

vscode_theme()

/home/gaby/.conda/envs/lab/lib/python3.13/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


HTML(value="<style>\n        .cell-output-ipywidget-background {\n                background: transparent !imp…

In [4]:
DATASETS_ROOT = Path("/run/media/gaby/GREY SSD/PostDoc/DATA/Fundus/")
DATASETS_PATH = {
    dataset: DATASETS_ROOT / folder
    for dataset, folder in {
        "GAVE-train": "GAVE-train",
        "MAPLES-DR": "MAPLES-DR",
        "FundusAV": "Fundus-AV",
        "HRF": "HRF",
        "LES-AV": "LES-AV",
        "INSPIRE": "INSPIRE",
        "DRIVE_train": "AV_DRIVE/training",
        "DRIVE_test": "AV_DRIVE/test",
    }.items()
}
RAW = [path / "1-images" for path in DATASETS_PATH.values()]
TOPO = [path / "3-topo" for path in DATASETS_PATH.values()]
AV = [
    {
        "gt": path / "2-av",
    }
    for path in DATASETS_PATH.values()
]


In [23]:
raw_samples = BranchDigraphDataset.discover_paths(RAW, TOPO, AV, dataset_name=list(DATASETS_PATH.keys()))
sample = raw_samples[160].process_in_memory(mask_optic_disc=False, resize_to=1024)

torch.Size([1, 3, 1280, 1280]) torch.Size([1, 3, 512, 512])


In [24]:
from fundus_vessels_toolkit.models.metrics.topologicial_fix import TopoFixesSummary, TopoFixes

fixes = TopoFixes.from_sample(sample)
fixes.inspect()

In [ ]:
summaries = {}
for raw, topo, av, dataset_name in zip(RAW, TOPO, AV, DATASETS_PATH.keys(), strict=True):
    diffs = []
    raw_samples = BranchDigraphDataset.discover_paths(raw, topo, av, dataset_name=dataset_name)
    for sample in tqdm(raw_samples, desc=f"Processing {dataset_name}"):
        sample = sample.process_in_memory(mask_optic_disc=False, resize_to=1024)
        diffs.extend(TopoFixes.from_sample(sample).diffs)
    summaries[dataset_name] = TopoFixesSummary(diffs)

pd.DataFrame({k: v.summary(per_image=True) for k, v in summaries.items()}).T

Processing GAVE-train:   0%|          | 0/50 [00:00<?, ?it/s]

torch.Size([1, 3, 1024, 1536]) torch.Size([1, 3, 512, 512])


Processing GAVE-train:   2%|▏         | 1/50 [00:01<01:01,  1.25s/it]

torch.Size([1, 3, 1024, 1536]) torch.Size([1, 3, 512, 512])


Processing GAVE-train:   4%|▍         | 2/50 [00:02<00:59,  1.25s/it]

torch.Size([1, 3, 1024, 1536]) torch.Size([1, 3, 512, 512])


... (746 outputs left out)

Processing DRIVE_test:  95%|█████████▌| 19/20 [00:20<00:01,  1.09s/it]

torch.Size([1, 3, 584, 565]) torch.Size([1, 3, 512, 512])


Processing DRIVE_test: 100%|██████████| 20/20 [00:21<00:00,  1.10s/it]


TypeError: TopoFixesSummary.summary() got an unexpected keyword argument 'per_image'

In [ ]:
dataset_lengths = {
    dataset_name: len(BranchDigraphDataset.discover_paths(raw, topo, av, dataset_name=dataset_name))
    for raw, topo, av, dataset_name in zip(RAW, TOPO, AV, DATASETS_PATH.keys(), strict=True)
}
pd.options.display.float_format = "{:,.2f}".format
pd.DataFrame({k: {s_k: s_v / dataset_lengths[k] for s_k, s_v in v.summary().items()} for k, v in summaries.items()})

,GAVE-train,MAPLES-DR,FundusAV,HRF,LES-AV,INSPIRE,DRIVE_train,DRIVE_test
erased,3.32,12.88,12.53,52.00,9.00,2.13,45.30,24.40
av_change,4.60,4.03,10.73,13.05,5.14,4.07,3.95,8.85
av_specified,0.16,5.50,3.64,20.39,4.59,0.47,6.25,3.75
disconnected,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
reconnection_out-of-view,0.10,0.83,0.44,0.54,0.23,0.47,0.80,0.70
reconnection_av_change,0.78,2.40,3.09,6.39,1.36,0.80,2.45,3.90
reconnection_crossing,7.60,31.64,5.05,24.07,1.73,1.93,5.60,7.30
reconnection_segmentation-fn,5.40,2.51,2.41,7.49,1.59,0.27,3.50,2.85


from fundus_toolkits import AVLabel
from fundus_vessels_toolkit.segment_to_graph.graph_simplification import simplify_passing_nodes
from fundus_vessels_toolkit.segment_to_graph.tree_topology import optimal_topology, transfer_topology

corrections = {}
for name, subset in dataset.split_by_dataset().items():
    reconnection = 0
    erased = 0
    av_changed = 0
    av_defined = 0

    for i in trange(len(subset)):
        sample = subset.get_sample(i, load_av_maps=True)
        graph = sample.graphes["gt"].copy()
        simplify_passing_nodes(graph, min_angle=100, inplace=True)
        tree = transfer_topology(sample.target_topologies, graph, allow_splits=True)

        reconnection += tree.branch_count - graph.branch_count

        src_av = graph.branch_attr["av"]
        fix_av = tree.branch_attr["av"][: len(src_av)]
        erased += (fix_av == AVLabel.BKG).sum()
        valid_fix = (fix_av == AVLabel.ART) | (fix_av == AVLabel.VEI)
        av_changed += (src_av[valid_fix] != fix_av[valid_fix]).sum()
        av_defined += (src_av[valid_fix] == AVLabel.BOTH).sum()

        break

    corrections[name] = {
        "reconnection": reconnection / len(subset),
        "erased": erased / len(subset),
        "av_changed": (av_changed - av_defined) / len(subset),
        "av_defined": av_defined / len(subset),
    }

    break

df = pd.DataFrame(corrections)
df.style.format("{:.1f}")